# 🧠 Machine Learning Data Preprocessing Pipeline
এই জুপিটার নোটবুকটিতে আমরা যা যা শিখেছি তা সম্পূর্ণ কোড আকারে সাজানো হয়েছে। আপনি এটি সরাসরি গুগল কোল্যাব (Google Colab) বা জুপিটার ল্যাবে রান করতে পারবেন।

### 🛠️ আমরা যা যা কভার করেছি:
- Pandas DataFrame তৈরি
- Train-Test Split দিয়ে ডেটা ভাগ করা
- SimpleImputer দিয়ে ফাঁকা ঘর (NaN) পূরণ
- OneHotEncoder দিয়ে লেখাকে সংখ্যায় রূপান্তর
- StandardScaler দিয়ে ফিচার স্কেলিং
- SelectKBest (f_classif) দিয়ে সেরা ফিচার সিলেকশন
- LogisticRegression মডেল ট্রেইনিং

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

print("✅ সকল লাইব্রেরি সফলভাবে ইমপোর্ট হয়েছে!")

### 📊 ধাপ ১: Pandas দিয়ে আমাদের সেই ভাঙাচোরা কাল্পনিক ডেটা তৈরি

In [ ]:
raw_data = {
    'Name': ['Abul', 'Babul', 'Kabul', 'Dabul', 'Ebul'],
    'Age': [30, np.nan, 40, 25, 35],  # বাবুলের বয়স NaN (ফাঁকা)
    'City': ['Dhaka', 'Ctg', 'Dhaka', 'Ctg', 'Dhaka'],
    'Bill': [20000, 5000, np.nan, 8000, 15000],  # কাবুলের বিল NaN (ফাঁকা)
    'Salary': [1, 0, 1, 0, 1]  # টার্গেট (১ = উচ্চ আয়, ০ = নিম্ন আয়)
}

df = pd.DataFrame(raw_data)
print("--- আমাদের মূল ডেটা টেবিল ---")
print(df)

### ✂️ ধাপ ২: ফিচার ও টার্গেট আলাদা করা এবং ডেটা ভাগ করা (Train-Test Split)

In [ ]:
# 'Name' কলামটি কোনো কাজের ফিচার নয়, তাই শুরুতেই বাদ
X = df[['Age', 'Bill', 'City']]
y = df['Salary']

# ডেটাকে ট্রেইনিং ও টেস্টিং সেটে ভাগ করা (৮০% ট্রেইন, ২০% টেস্ট)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train এর সাইজ: {X_train.shape}")
print(f"X_test এর সাইজ: {X_test.shape}")

### 🚰 ধাপ ৩: ফুল অটোমেটিক পাইপলাইন (Pipeline & ColumnTransformer) তৈরি

In [ ]:
# ১. সংখ্যার কলামের জন্য সাব-পাইপলাইন (ইম্পুটার + স্কেলার)
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# ২. লেখার কলামের জন্য সাব-পাইপলাইন (ওয়ান-হট এনকোডার)
categorical_pipeline = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# ৩. কলাম ট্রান্সফরমার দিয়ে প্রসেসর তৈরি
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, ['Age', 'Bill']),
    ('cat', categorical_pipeline, ['City'])
])

# ৪. ফাইনাল জুস ফ্যাক্টরি পাইপলাইন (পরিষ্কারক + ফিচার সিলেক্টর + এআই মডেল)
auto_machine = Pipeline(steps=[
    ('cleaner', preprocessor),
    ('feature_selector', SelectKBest(score_func=f_classif, k=2)), # সেরা ২টি ফিচার বাছবে
    ('classifier', LogisticRegression())
])

print("⚙️ অটোমেটিক পাইপলাইন আর্কিটেকচার তৈরি সফল হয়েছে!")

### 🚀ধাপ ৪: মডেল ট্রেইনিং (Fit) এবং নতুন ডেটায় প্রেডিকশন

In [ ]:
# পাইপলাইন চালু করে মডেলকে ট্রেইন করা
auto_machine.fit(X_train, y_train)
print("🎉 মডেল সফলভাবে ট্রেইন বা Fit হয়েছে!")

# X_test ডেটার ওপর প্রেডিকশন পরীক্ষা করা
predictions = auto_machine.predict(X_test)
print("\n--- টেস্টিং ডেটার ওপর মডেলের প্রেডিকশন ---")
print("আসল উত্তর (y_test):", y_test.values)
print("এআই-এর উত্তর (predictions):", predictions)